Sel 1: muat data


In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from review_nlp.preprocessing import clean_text, preprocess
from review_nlp.preprocessing.slang_normalizer import load_slang

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "data/raw/com.whatsapp_2026-08-19_2026-11-17.parquet")
used = df[df["exclude_reason"].isna()].copy()
print("Ulasan dipakai:", len(used))

Ulasan dipakai: 5768


Sel 2: contoh sebelum dan sesudah

In [2]:
for raw in used["text_raw"].sample(10, random_state=42):
    print("SEBELUM:", raw)
    print("SESUDAH:", preprocess(raw), "\n")

SEBELUM: wa gua msih di blok kpn bukanya woi?
SESUDAH: whatsapp gua masih di blok kapan bukannya woi 

SEBELUM: bagus tapi punya ku gak ada filter kayak yang lain
SESUDAH: bagus tapi punya ku tidak ada filter kayak yang lain 

SEBELUM: meta aneh.. tgl 21 september ini akun aku kena band. padahal gak berbuat yang aneh". cuman neruskan pesan grup kerjaan. bukan melakukan yang aneh aneh. melakukan bandingpun tetap gak berguna.
SESUDAH: meta aneh tanggal 21 september ini akun aku kena band padahal tidak berbuat yang aneh cuman neruskan pesan grup kerjaan bukan melakukan yang aneh aneh melakukan bandingpun tetap tidak berguna 

SEBELUM: oh my God
SESUDAH: oh my god 

SEBELUM: ok sangat membantu
SESUDAH: ok sangat membantu 

SEBELUM: buruk apklikasi nya
SESUDAH: buruk apklikasi nya 

SEBELUM: 𝗴𝗮 𝗮𝗱𝗮 𝗲𝗳𝗲𝗸 di status sama efek vc
SESUDAH: tidak ada efek di status sama efek vc 

SEBELUM: saya mau bikin status wa tapi gak bisa mengeteg teman kita
SESUDAH: saya mau bikin status whatsapp tapi tidak

In [3]:
gaya = used["text_raw"].map(lambda t: any(0x1D400 <= ord(c) <= 0x1D7FF for c in t))
print("Ulasan dengan huruf bergaya:", int(gaya.sum()), f"({gaya.mean():.2%})")
used.loc[gaya, "text_raw"].head(10)

Ulasan dengan huruf bergaya: 9 (0.16%)


611      𝐚𝐩𝐥𝐢𝐤𝐚𝐬𝐢𝐧𝐲𝐚 𝐛𝐚𝐠𝐮𝐬 𝐭𝐚𝐩𝐢 𝐤𝐨 𝐬𝐞𝐭𝐢𝐚𝐩 𝐛𝐢𝐤𝐢𝐧 𝐬𝐭𝐨𝐫𝐲 𝐣...
4051     𝐛𝐢𝐤𝐢𝐧 𝐛𝐢𝐬𝐚 𝐥𝐢𝐡𝐚𝐭 𝐬𝐢𝐚𝐩𝐚 𝐲𝐚𝐧𝐠 𝐟𝐨𝐫𝐰𝐚𝐫𝐝 𝐩𝐞𝐬𝐚𝐧 𝐩𝐞𝐬𝐚...
4065     𝘄𝗵𝗮𝘁𝘀𝗮𝗽𝗽 𝘀𝗸𝗿𝗻𝗴 𝗯𝗮𝗻𝘆𝗮𝗸 𝗸𝗲𝗻𝗱𝗮𝗹𝗮 𝗻𝘆𝗮, 𝗴𝗮 𝗯𝘂𝗮𝘁 𝗮𝗽𝗮...
4143                    𝗴𝗮 𝗮𝗱𝗮 𝗲𝗳𝗲𝗸 di status sama efek vc
6755                            ᥲ⍴ᥣіkᥲsі ᥒᥡᥲ sᥲᥒgᥲ𝗍 ᥒᥡᥲᥒᥲᥒ
8986     𝒔𝒆𝒍𝒂𝒍𝒖𝒖 𝒌𝒆𝒏𝒂 𝒎𝒂𝒔𝒂 𝒕𝒊𝒏𝒋𝒂𝒖 𝒑𝒂𝒅𝒂𝒉𝒂𝒍 𝒈𝒌 𝒂𝒅𝒂 𝒎𝒆𝒍𝒂𝒏𝒈...
9437                            𝒂𝒑𝒍𝒊𝒌𝒂𝒔𝒊 𝑰𝑵𝒊 𝒍𝒖𝒎𝒂𝒚𝒂𝒏 𝒃𝒂𝒈𝒊𝒔
10154                                 𝙨𝙖𝙣𝙜𝙖𝙩 𝙨𝙚𝙧𝙪 𝙬𝙖𝙨𝙖𝙥 𝙮𝙖
10188                                   𝒔𝒂𝒏𝒈𝒂𝒕 𝒃𝒂𝒊𝒌 𝒔𝒆𝒌𝒂𝒍𝒊
Name: text_raw, dtype: str

In [4]:
print(used["lang"].value_counts().head(8))
inggris = used[used["lang"] == "en"]
print("\nLabel bahasa 'en' tapi dipakai:", len(inggris), f"({len(inggris) / len(used):.1%})")
inggris[["rating", "text_raw"]].sample(min(15, len(inggris)), random_state=1)

lang
id    4959
tl     433
sw      60
so      53
en      39
et      25
de      21
no      20
Name: count, dtype: int64

Label bahasa 'en' tapi dipakai: 39 (0.7%)


,rating,text_raw
455,1,"APALAH INI WA, DARI KEMARIN AKUN WA GUA HILANG..."
7750,1,APLIKASI TOLOL UDA MAH BUAT SW LAMA BURIK LAGI...
674,5,tolong tambahin fitur chat 1× liatt min
6175,1,"of9_ a tf c5 c '_ cfcfo c f ""5 of,, of, o9"" _l..."
7164,1,"APK MACAM APA INI HADEHHH, MASAA CHATAN SAMA T..."
7710,5,wa tolong tambahin fitur foto live dong 🤭🤭🤭
6632,5,login wa cepat
9312,3,INI KENAPA YAA FITUR EDIT ILANGGG
5517,5,wis mantap 100 but watshap
7047,3,ancok update trs


In [5]:
campuran = used[used["lang"].isin(["tl", "sw", "so", "et", "no", "de"])]
print("Berlabel tl/sw/so/et/no/de tapi dipakai:", len(campuran), f"({len(campuran) / len(used):.1%})")
campuran[["lang", "text_raw"]].sample(20, random_state=3)

Berlabel tl/sw/so/et/no/de tapi dipakai: 612 (10.6%)


,lang,text_raw
7686,no,Watsapp error payah
2518,tl,Ko wasap aku gak bisa d guna kan
4223,de,Bagusssss Buat LDR Kvdidb
1083,tl,aplikasinya sangat bagus
4446,tl,saya sangat butuh aplikasi ini.karna sangat me...
2485,tl,memang paling bagus
6619,sw,SANGAT BAIK UNTUK KOMUNIKASI
8708,tl,baguss kalo wa apdte yg semua langsung yaa
6846,tl,apaan tiba tiba di band no wa saya dan gak bis...
2312,tl,WhatsApp ini sangat bagus


### Temuan: deteksi bahasa
- Kolom `lang` hasil `langdetect` **tidak andal untuk teks gaul/pendek**: sekitar 10% ulasan
  berbahasa Indonesia terdeteksi sebagai Tagalog, Swahili, Somali, dll.
- Karena itu filter hanya membuang bahasa yang **jelas asing** (Inggris, Arab, dll.) dengan
  keyakinan tinggi (>= 0,90). Label `tl/sw/so/et/no/de` sengaja tidak dibuang.
- Ulasan berlabel `en` yang lolos hanya 0,7%, mayoritas campuran Inggris-Indonesia (sah).
- Sebagian ulasan memuat bahasa daerah dan gaul; ini dipertahankan dan menjadi keterbatasan.

Sel 3: kandidat kata slang

In [6]:
slang = load_slang()
counter = Counter(tok for t in used["text_raw"] for tok in clean_text(t).split())

kandidat = [(w, c) for w, c in counter.most_common(500) if w not in slang]
print("Top 150 token yang belum ada di kamus:")
display(pd.DataFrame(kandidat[:150], columns=["token", "frekuensi"]))

pendek = [(w, c) for w, c in kandidat if len(w) <= 5][:100]
print("Token pendek (<= 5 huruf), calon singkatan:")
display(pd.DataFrame(pendek, columns=["token", "frekuensi"]))

Top 150 token yang belum ada di kamus:


,token,frekuensi
0,di,2151
1,saya,1874
2,bisa,1591
3,whatsapp,1221
4,tidak,1092
...,...,...
145,setiap,81
146,kaya,81
147,masalah,81
148,gitu,81


Token pendek (<= 5 huruf), calon singkatan:


,token,frekuensi
0,di,2151
1,saya,1874
2,bisa,1591
3,tidak,1092
4,nya,994
...,...,...
95,kode,90
96,tiba2,89
97,mudah,89
98,pakai,88


Sel 4: tambah kata ke kamus, lalu muat ulang

In [7]:
load_slang.cache_clear()
print("Jumlah entri kamus:", len(load_slang()))

Jumlah entri kamus: 61


In [8]:
cek = ["ndak", "nda", "kagak", "kaga", "ora", "iso", "gak", "gk", "bgt", "banget"]
{w: counter[w] for w in cek}

{'ndak': 7,
 'nda': 1,
 'kagak': 8,
 'kaga': 33,
 'ora': 5,
 'iso': 1,
 'gak': 475,
 'gk': 190,
 'bgt': 93,
 'banget': 323}

Sel 5: efek preprocessing

In [9]:
used["clean"] = used["text_raw"].map(preprocess)

vocab_raw = len({t for s in used["text_raw"] for t in clean_text(s).split()})
vocab_norm = len({t for s in used["clean"] for t in s.split()})
print("Ukuran kosakata sebelum normalisasi slang:", vocab_raw)
print("Ukuran kosakata sesudah normalisasi slang:", vocab_norm)

kosong = used[used["clean"].str.len() == 0]
print("Teks kosong setelah preprocessing:", len(kosong))
kosong[["text_raw"]].head(10)

Ukuran kosakata sebelum normalisasi slang: 7281
Ukuran kosakata sesudah normalisasi slang: 7220
Teks kosong setelah preprocessing: 0


,text_raw


Sel 6: kata khas negatif vs positif

In [10]:
neg = Counter(tok for t in used.loc[used["rating"] <= 2, "clean"] for tok in t.split())
pos = Counter(tok for t in used.loc[used["rating"] >= 4, "clean"] for tok in t.split())
vocab = [w for w in set(neg) | set(pos) if neg[w] + pos[w] >= 20]
n_neg, n_pos = sum(neg.values()), sum(pos.values())
skor = {
    w: np.log((neg[w] + 1) / (n_neg + 1)) - np.log((pos[w] + 1) / (n_pos + 1))
    for w in vocab
}
urut = sorted(skor.items(), key=lambda x: x[1])
pd.DataFrame(
    {
        "khas_positif": [w for w, _ in urut[:20]],
        "khas_negatif": [w for w, _ in urut[-20:][::-1]],
    }
)

,khas_positif,khas_negatif
0,mantap,gajelas
1,terbaik,sampah
2,cocok,kecewa
3,menarik,tolol
4,membantu,emang
5,senang,merugikan
6,keren,bukannya
7,tambahin,melanggar
8,bermanfaat,apaan
9,oke,lu


Sel 7: apakah emoji membawa sentimen

In [11]:
from review_nlp.preprocessing.cleaner import EMOJI_RE

used["punya_emoji"] = used["text_raw"].map(lambda t: bool(EMOJI_RE.search(t)))
print("Proporsi ulasan beremoji:", round(used["punya_emoji"].mean(), 3))
pd.crosstab(used["punya_emoji"], used["rating"], normalize="index").round(3)

Proporsi ulasan beremoji: 0.074


rating,1,2,3,4,5
punya_emoji,,,,,
False,0.332,0.072,0.088,0.089,0.420
True,0.259,0.064,0.118,0.090,0.469


### Temuan: preprocessing
- Normalisasi slang hanya mengurangi kosakata dari 7.226 menjadi 7.170 (0,8%).
  Kamus kecil cukup; TF-IDF dengan bigram sudah menangkap variasi yang sering muncul.
- 6 ulasan (0,1%) kosong setelah dibersihkan dan dibuang saat pelatihan.
- Huruf bergaya Unicode (mis. tulisan tebal) hanya 0,16% dan kini dinormalisasi.
- Ulasan beremoji 7,4%; sedikit condong positif tetapi masih dalam batas ketidakpastian
  sampel, sehingga emoji tetap dibuang. Calon perbaikan: petakan emoji menjadi kata sentimen.